<a href="https://colab.research.google.com/github/kartik2006-del/biopapers-rag/blob/main/BioPapers_RAG_Data_Collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PubMed Data Fetching

In [1]:

import requests
import pandas as pd
import xml.etree.ElementTree as ET




In [2]:
url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"

params = {
    "db": "pubmed",
    "term": '"CRISPR"[Title/Abstract] AND "gene editing"[Title/Abstract]',
    "retmax": 5,
    "retmode": "json"
}

response = requests.get(url, params=params, timeout=30)

print("Status code:", response.status_code)
print("PMIDs:", response.json()["esearchresult"]["idlist"])

Status code: 200
PMIDs: ['42906700', '42904570', '42902180', '42899764', '42899205']


In [3]:
fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

fetch_params = {
    "db": "pubmed",
    "id": ",".join(response.json()["esearchresult"]["idlist"]),
    "rettype": "abstract",
    "retmode": "xml"
}

fetch_response = requests.get(
    fetch_url,
    params=fetch_params,
    timeout=30
)

print("Status code:", fetch_response.status_code)
print(fetch_response.text[:1000])

Status code: 200
<?xml version="1.0" ?>
<!DOCTYPE PubmedArticleSet PUBLIC "-//NLM//DTD PubMedArticle, 1st January 2025//EN" "https://dtd.nlm.nih.gov/ncbi/pubmed/out/pubmed_250101.dtd">
<PubmedArticleSet>
<PubmedArticle><MedlineCitation Status="Publisher" Owner="NLM"><PMID Version="1">42906700</PMID><DateRevised><Year>2026</Year><Month>10</Month><Day>22</Day></DateRevised><Article PubModel="Print-Electronic"><Journal><ISSN IssnType="Electronic">1873-2259</ISSN><JournalIssue CitedMedium="Internet"><PubDate><Year>2026</Year><Month>Oct</Month><Day>21</Day></PubDate></JournalIssue><Title>Plant science : an international journal of experimental plant biology</Title><ISOAbbreviation>Plant Sci</ISOAbbreviation></Journal><ArticleTitle>The conserved function of bHLH121 and overexpression of N-terminal bHLH121 fragment suppresses iron deficiency response in Arabidopsis.</ArticleTitle><Pagination><StartPage>113540</StartPage><MedlinePgn>113540</MedlinePgn></Pagination><ELocationID EIdType="doi" Va

In [4]:
root = ET.fromstring(fetch_response.content)

article = root.find(".//PubmedArticle")

title = article.findtext(
    ".//ArticleTitle",
    default="Title not found"
)

print("First paper title:")
print(title)

First paper title:
The conserved function of bHLH121 and overexpression of N-terminal bHLH121 fragment suppresses iron deficiency response in Arabidopsis.


In [5]:
articles = root.findall(".//PubmedArticle")

for article in articles:
    pmid = article.findtext(".//PMID", default="N/A")
    title = article.findtext(".//ArticleTitle", default="N/A")

    print("PMID:", pmid)
    print("Title:", title)
    print("-" * 80)

PMID: 42906700
Title: The conserved function of bHLH121 and overexpression of N-terminal bHLH121 fragment suppresses iron deficiency response in Arabidopsis.
--------------------------------------------------------------------------------
PMID: 42904570
Title: Synthetic Biology Toolbox for Lactiplantibacillus plantarum: Engineering Next-Generation Probiotic Cell Factories.
--------------------------------------------------------------------------------
PMID: 42902180
Title: Periplocin targets low density lipoprotein receptor-related protein 4 and protects against osteoarthritis.
--------------------------------------------------------------------------------
PMID: 42899764
Title: Nano Delivery Systems for In Vivo CRISPR/Cas-Based Gene Therapy: From Vectors to Clinical Outcomes.
--------------------------------------------------------------------------------
PMID: 42899205
Title: Genome editing beyond transgenes: DNA-free technologies in crop improvement.
-------------------------------